## Last.fm top artists by genre

In [18]:
import os

import requests
from dotenv import find_dotenv, load_dotenv


API_URL = "https://ws.audioscrobbler.com/2.0/"
load_dotenv(find_dotenv(usecwd=True), override=True)  # finds ../.env at the repo root (gitignored)
API_KEY = os.environ["LASTFM_API_KEY"]

# Last.fm tag names; R&B is tagged "rnb" there
GENRES = {
    "rock": "rock",
    "pop": "pop",
    "hip-hop": "hip-hop",
    "R&B": "rnb",
    "country": "country",
}
N_ARTISTS = 10
POOL_SIZE = 2 * N_ARTISTS  # spare artists in case some lack enough songs on Genius

In [19]:
def top_artists_for_tag(tag: str, limit: int = POOL_SIZE) -> list[str]:
    response = requests.get(
        API_URL,
        params={
            "method": "tag.gettopartists",
            "tag": tag,
            "limit": limit,
            "api_key": API_KEY,
            "format": "json",
        },
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    if "error" in data:
        raise RuntimeError(f"Last.fm error {data['error']}: {data['message']}")
    return [artist["name"] for artist in data["topartists"]["artist"]]


artists_by_genre = {genre: top_artists_for_tag(tag) for genre, tag in GENRES.items() if genre == "pop"}  # R&B is handled separately due to the tag name difference

for genre, artists in artists_by_genre.items():
    print(f"{genre}: {', '.join(artists)}\n")

pop: Ariana Grande, Sabrina Carpenter, Olivia Rodrigo, ADÉLA, Lady Gaga, Michael Jackson, Katy Perry, Britney Spears, Rihanna, Madonna, Bruno Mars, Justin Bieber, Dua Lipa, Billie Eilish, Miley Cyrus, Addison Rae, Doja Cat, Taylor Swift, Charli xcx, Chappell Roan



## Get top songs for each artist 

In [20]:
import random
import re
import time
from pathlib import Path

import lyricsgenius
import pandas as pd

SONGS_PER_ARTIST = 5
MAX_RELEASE_YEAR = 2024  # only songs released in or before this year
OUTPUT_DIR = Path(find_dotenv(usecwd=True)).parent / "outputs"
SKIPPED_PATH = OUTPUT_DIR / "skipped_artists.csv"


def lyrics_path(genre: str) -> Path:
    return OUTPUT_DIR / f"lyrics_{re.sub(r'[^a-z0-9]+', '_', genre.lower()).strip('_')}.csv"


def release_year(song: dict) -> int | None:
    return (song.get("release_date_components") or {}).get("year")


EXCLUDED_TITLE_WORDS = ["Remix", "Cover", "Edit", "Live", "Acapella", "Instrumental", "Karaoke"]
EXCLUDED_TITLE_RE = re.compile(
    r"\b(" + "|".join(map(re.escape, EXCLUDED_TITLE_WORDS)) + r")\b", re.IGNORECASE
)


# Throttled on purpose: pause between requests, time out, retry with backoff
# Not subclassed: lyricsgenius only sends the token when the class is named "Genius"
genius = lyricsgenius.Genius(
    os.environ["GENIUS_ACCESS_TOKEN"],
    sleep_time=1.5,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True,
)

_is_lyrics = genius._result_is_lyrics
genius._result_is_lyrics = lambda song: (
    _is_lyrics(song)
    and not EXCLUDED_TITLE_RE.search(song["title"])
    # songs without a known release date can't be confirmed, so they are skipped
    and (release_year(song) or MAX_RELEASE_YEAR + 1) <= MAX_RELEASE_YEAR
)


In [21]:
def fetch_artist_songs(artist_name: str, genre: str) -> list[dict] | None:
    """Returns the artist's songs, or None if the artist is not found on Genius."""
    artist = genius.search_artist(
        artist_name,
        max_songs=SONGS_PER_ARTIST,
        sort="popularity",
        get_full_info=False,
        allow_name_change=False,
    )
    if artist is None:
        return None
    return [
        {
            "genre": genre,
            "artist": artist_name,
            "title": song.title,
            "release_year": release_year(song._body),
            "url": song.url,
            "lyrics": song.lyrics,
        }
        for song in artist.songs
    ]


def load_records(path: Path) -> list[dict]:
    return pd.read_csv(path).to_dict("records") if path.exists() else []


# Resume: keep artists accepted or skipped by a previous run
OUTPUT_DIR.mkdir(exist_ok=True)
rows = [row for genre in artists_by_genre for row in load_records(lyrics_path(genre))]
skipped = load_records(SKIPPED_PATH)

for genre, artists in artists_by_genre.items():
    accepted = {row["artist"] for row in rows if row["genre"] == genre}
    skipped_names = {row["artist"] for row in skipped if row["genre"] == genre}

    for artist_name in artists:
        if len(accepted) >= N_ARTISTS:
            break
        if artist_name in accepted or artist_name in skipped_names:
            continue

        print(f"{genre}: {artist_name}")
        try:
            artist_rows = fetch_artist_songs(artist_name, genre)
        except Exception as error:  # network error: not skipped, retried next run
            print(f"  failed: {error}")
            continue

        if artist_rows is None or len(artist_rows) < SONGS_PER_ARTIST:
            reason = (
                "not found on Genius"
                if artist_rows is None
                else f"only {len(artist_rows)} valid songs"
            )
            print(f"  skipped: {reason}")
            skipped.append({"genre": genre, "artist": artist_name, "reason": reason})
            pd.DataFrame(skipped).to_csv(SKIPPED_PATH, index=False)
        else:
            rows += artist_rows
            accepted.add(artist_name)
            genre_rows = [row for row in rows if row["genre"] == genre]
            pd.DataFrame(genre_rows).to_csv(lyrics_path(genre), index=False)
        time.sleep(random.uniform(2, 5))

    if len(accepted) < N_ARTISTS:
        print(f"WARNING: {genre} has only {len(accepted)}/{N_ARTISTS} artists")

songs = pd.DataFrame(rows)
songs.groupby("genre")["artist"].nunique()

pop: Ariana Grande
pop: Sabrina Carpenter
pop: Olivia Rodrigo
pop: ADÉLA
  skipped: only 3 valid songs
pop: Lady Gaga
pop: Michael Jackson
pop: Katy Perry
pop: Britney Spears
pop: Rihanna
pop: Madonna
pop: Bruno Mars


genre
pop    10
Name: artist, dtype: int64